# 🎙️ Voice & CLI-Driven System Virtual Assistant — Walkthrough

This notebook is a guided tour of the assistant that lives in this folder. It explains
**how each layer works** and runs the pieces that are safe to execute inside a notebook.

| Layer | Module | What it does |
|---|---|---|
| Speech & NLP | `assistant/speech.py` | time-aware greetings, offline TTS, microphone STT |
| Intent routing | `assistant/commands.py` | regex intent router that turns an utterance into an action |
| Knowledge | `assistant/knowledge.py` | Wikipedia summaries, Google search, YouTube playback |
| Live data | `assistant/live_data.py` | weather + top news headlines (keyless providers) |
| System automation | `assistant/system_tools.py` | clipboard, screenshots, battery, media, power controls |
| Orchestration | `assistant/core.py`, `assistant/cli.py` | the assistant object and its command line |

> **Note on safety** — screenshots, media playback and power commands touch the host
> machine, so this notebook demonstrates them through injected test doubles rather than
> firing them for real. The interactive assistant (`python main.py`) is where they run.

In [1]:
import sys
from datetime import datetime
from pathlib import Path

# The notebook lives next to the package, so the project folder is the import root.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assistant").is_dir():
    PROJECT_ROOT = PROJECT_ROOT / "Voice_CLI_Virtual_Assistant"
sys.path.insert(0, str(PROJECT_ROOT))

import assistant
from assistant import commands, knowledge, live_data, system_tools
from assistant.config import AssistantConfig, load_config
from assistant.core import VirtualAssistant
from assistant.speech import greeting_for, spoken_date, spoken_time

print(f"assistant package v{assistant.__version__}")
print(f"python {sys.version.split()[0]}")

assistant package v1.0.0
python 3.14.6


## 1. Speech layer — time-aware greetings

`greeting_for()` is a pure function of the clock, which keeps it trivially testable.
The same helper drives the greeting spoken at start-up.

In [2]:
for hour in (8, 13, 19, 23):
    moment = datetime(2026, 8, 1, hour, 0)
    print(f"{moment:%H:%M} -> {greeting_for(moment)}")

print()
print("spoken time:", spoken_time(datetime(2026, 8, 1, 9, 5)))
print("spoken date:", spoken_date(datetime(2026, 8, 1)))

08:00 -> Good morning
13:00 -> Good afternoon
19:00 -> Good evening
23:00 -> Good night

spoken time: 9:05 AM
spoken date: Saturday, 1 August 2026


Text-to-speech (`pyttsx3`) and speech-to-text (`SpeechRecognition`) are imported
**lazily**. If the host has no audio driver or no microphone, the engine quietly falls
back to printing responses and reading typed input — the assistant never crashes because
of missing hardware.

In [3]:
from assistant.speech import SpeechEngine

engine = SpeechEngine(load_config(), enable_tts=False, enable_stt=False)
print("TTS available :", engine.tts_available)
print("STT available :", engine.stt_available)
print()
engine.speak("This is what a spoken response looks like in text mode.")

TTS available : False
STT available : False

assistant > This is what a spoken response looks like in text mode.


'This is what a spoken response looks like in text mode.'

## 2. Intent routing

Every utterance — typed or transcribed — is normalised (lower-cased, punctuation removed,
wake word stripped) and then matched against an **ordered** list of regex intents.

Ordering matters: `what is the time` must beat the catch-all `what is <topic>` Wikipedia
lookup, and `play file <path>` must beat `play <song>`.

In [4]:
print(repr(commands.normalise("Hey Capsule, what's the TIME??", wake_word="capsule")))
print(repr(commands.normalise("CAPSULE: read me the news!", wake_word="capsule")))
# A wake word in the middle of a sentence is left alone.
print(repr(commands.normalise("what is a capsule", wake_word="capsule")))

"what's the time"
': read me the news'
'what is a capsule'


In [5]:
router = commands.CommandRouter()

utterances = [
    "hello there",
    "what is the time",
    "todays date",
    "weather in Mumbai",
    "read me the news",
    "battery status",
    "take a screenshot",
    "read my clipboard",
    "system info",
    "play file ~/Music/song.mp3",
    "play shape of you",
    "who is Alan Turing",
    "google best ML projects",
    "shutdown",
    "make me a sandwich",
]

print(f"{'utterance':<32} -> intent")
print("-" * 52)
for text in utterances:
    result = router.match(commands.normalise(text, "capsule"))
    print(f"{text:<32} -> {result[0].name if result else '(unmatched)'}")

utterance                        -> intent
----------------------------------------------------
hello there                      -> greet
what is the time                 -> time
todays date                      -> date
weather in Mumbai                -> weather
read me the news                 -> news
battery status                   -> battery
take a screenshot                -> screenshot
read my clipboard                -> clipboard
system info                      -> system
play file ~/Music/song.mp3       -> open_media
play shape of you                -> youtube
who is Alan Turing               -> wikipedia
google best ML projects          -> google
shutdown                         -> power
make me a sandwich               -> (unmatched)


## 3. Live-data parsers

Network access is confined to the `fetch_*` helpers; the payload parsers are pure
functions, so they can be exercised here with fixtures — no network, no API key.

In [6]:
wttr_payload = {
    "current_condition": [{
        "temp_C": "29", "FeelsLikeC": "33", "humidity": "74",
        "windspeedKmph": "11", "weatherDesc": [{"value": "Partly cloudy"}],
    }]
}
print(live_data.parse_wttr_payload(wttr_payload, "Jamshedpur").describe())

openweather_payload = {
    "name": "Jamshedpur",
    "weather": [{"description": "light rain"}],
    "main": {"temp": 27.4, "feels_like": 30.1, "humidity": 88},
    "wind": {"speed": 5},  # m/s, converted to km/h by the parser
}
print(live_data.parse_openweather_payload(openweather_payload, "Jamshedpur").describe())

Partly cloudy in Jamshedpur. It is 29 degrees Celsius and feels like 33, with 74% humidity and 11 kilometres per hour of wind.
Light rain in Jamshedpur. It is 27 degrees Celsius and feels like 30, with 88% humidity and 18 kilometres per hour of wind.


In [7]:
rss = '''<?xml version="1.0"?>
<rss version="2.0"><channel>
  <item><title>First headline</title></item>
  <item><title>Second headline</title></item>
  <item><title>Third headline</title></item>
</channel></rss>'''

print(live_data.format_headlines(live_data.parse_news_rss(rss, limit=3)))
print()
# A malformed feed degrades to an empty list rather than raising.
print("malformed feed ->", live_data.parse_news_rss("not xml at all"))

Here are the top 3 headlines.
1. First headline
2. Second headline
3. Third headline

malformed feed -> []


### Optional: live providers

Both providers work **without an API key**: weather falls back to `wttr.in` and news to the
Google News RSS feed. Export `OPENWEATHER_API_KEY` / `NEWSAPI_KEY` to use those services
instead. The cell below is wrapped so the notebook still runs offline (for example in CI).

In [8]:
config = load_config()
print(f"city: {config.city} | OpenWeather key set: {bool(config.openweather_api_key)} "
      f"| NewsAPI key set: {bool(config.newsapi_key)}")

try:
    print()
    print(live_data.fetch_weather(config.city, config))
    print()
    print(live_data.format_headlines(live_data.fetch_headlines(config, limit=3)))
except Exception as exc:  # offline execution keeps the notebook runnable
    print(f"live lookup skipped: {exc}")

city: Jamshedpur | OpenWeather key set: False | NewsAPI key set: False



Mist in Jamshedpur. It is 26 degrees Celsius and feels like 30, with 95% humidity and 4 kilometres per hour of wind.



Here are the top 3 headlines.
1. 'Want To Forgive Them': PM Modi On Students Who Abused Him at Jantar Mantar - NDTV
2. Spain deploys military to Ceuta after migrant surge: What we know - Al Jazeera
3. 'Pak-trained Mujahideen have turned their guns inwards': India slams Islamabad over PoK crackdown - The Times of India


## 4. Knowledge integrations

URL builders are pure functions, so the exact URL that would be opened can be asserted
without launching a browser.

In [9]:
print(knowledge.google_search_url("best ML projects"))
print(knowledge.youtube_search_url("lofi beats"))
print(knowledge.google_search_url("c++ & python"))  # special characters are escaped

# `open_in_browser` takes an injectable opener, which is how the tests stay headless.
opened = []
knowledge.play_on_youtube("shape of you", opener=opened.append)
print("would open:", opened)

https://www.google.com/search?q=best+ML+projects
https://www.youtube.com/results?search_query=lofi+beats
https://www.google.com/search?q=c%2B%2B+%26+python
would open: ['https://www.youtube.com/results?search_query=shape+of+you']


## 5. System automation & telemetry

`psutil`, `pyautogui` and `pyperclip` are all optional. Each helper returns a readable
message instead of raising when its dependency (or the hardware) is missing.

In [10]:
print("host:", system_tools.system_summary())
print("screenshot name:", system_tools.screenshot_filename(datetime(2026, 8, 1, 14, 30, 5)))

battery = system_tools.battery_status()
print("battery:", battery.describe() if battery else "no battery detected on this host")

# Formatting is a pure function, so the description is verifiable with a fixture.
sample = system_tools.BatteryStatus(percent=76.4, plugged=False, seconds_left=5400)
print("sample :", sample.describe())

host: Darwin 25.5.0 on arm64, running Python 3.14.
screenshot name: screenshot_20260801_143005.png
battery: The battery is at 76 percent and on battery. About 7 hours and 41 minutes remaining.
sample : The battery is at 76 percent and on battery. About 1 hours and 30 minutes remaining.


## 6. Safety model for destructive actions

Shutdown / restart / logout sit behind **two independent gates**:

1. they are disabled unless the user opts in (`--allow-power`, or `ASSISTANT_ALLOW_POWER=1`);
2. even then, the user must type the action name to confirm.

The cell below proves both gates hold, using a recording runner so nothing is executed.

In [11]:
executed = []

# Gate 1 - opt-in missing: the command is described, never run.
ok, message = system_tools.run_power_command(
    "shutdown", AssistantConfig(allow_power_commands=False),
    confirm=lambda action: True, runner=executed.append,
)
print(f"gate 1 -> ran={ok}\n         {message}\n")

# Gate 2 - opted in, but confirmation declined.
ok, message = system_tools.run_power_command(
    "restart", AssistantConfig(allow_power_commands=True),
    confirm=lambda action: False, runner=executed.append,
)
print(f"gate 2 -> ran={ok}\n         {message}\n")

# Both gates satisfied - only now is a command issued (to the recording runner).
ok, message = system_tools.run_power_command(
    "logout", AssistantConfig(allow_power_commands=True),
    confirm=lambda action: True, runner=executed.append,
)
print(f"both   -> ran={ok}\n         {message}\n")
print("commands actually dispatched:", executed)

gate 1 -> ran=False
         Power commands are disabled. Restart me with --allow-power to enable 'shutdown' (it would run: osascript -e tell app "System Events" to shut down).

gate 2 -> ran=False
         Cancelled the restart request.

both   -> ran=True
         Logout command issued.

commands actually dispatched: [['osascript', '-e', 'tell app "System Events" to log out']]


## 7. Driving the whole assistant

`VirtualAssistant` accepts injected collaborators, so a complete end-to-end conversation
can be replayed here without a microphone, a browser or a real power command.

In [12]:
class RecordingSpeech:
    """Stands in for the speech engine and records what would be spoken."""

    def __init__(self):
        self.spoken, self.enable_stt, self.stt_available = [], False, False

    def speak(self, text):
        self.spoken.append(text)
        return text


opened_urls = []
speech = RecordingSpeech()
bot = VirtualAssistant(
    config=AssistantConfig(city="Jamshedpur", wake_word="capsule"),
    speech=speech,
    opener=opened_urls.append,
    confirm=lambda action: False,
)

conversation = [
    "Hey Capsule, hello!",
    "what is the time",
    "google machine learning roadmap",
    "play lofi beats",
    "shutdown",
    "make me a sandwich",
    "bye",
]

for line in conversation:
    response = bot.respond(line)
    flag = "" if response.handled else "   <- not understood"
    print(f"you       > {line}")
    print(f"assistant > {response.text}{flag}")
    if response.should_exit:
        print("[session ended]")
        break
    print()

print()
print("URLs the assistant would have opened:")
for url in opened_urls:
    print("  ", url)

you       > Hey Capsule, hello!
assistant > Good morning! How can I help you?

you       > what is the time
assistant > The time is 1:18 AM.

you       > google machine learning roadmap
assistant > Searching Google for machine learning roadmap. Opened https://www.google.com/search?q=machine+learning+roadmap

you       > play lofi beats
assistant > Opening YouTube results for lofi beats. Opened https://www.youtube.com/results?search_query=lofi+beats

you       > shutdown
assistant > Power commands are disabled. Restart me with --allow-power to enable 'shutdown' (it would run: osascript -e tell app "System Events" to shut down).

you       > make me a sandwich
assistant > I did not understand that. Say 'help' to see what I can do.   <- not understood

you       > bye
assistant > Goodbye! Shutting down the assistant.
[session ended]

URLs the assistant would have opened:
   https://www.google.com/search?q=machine+learning+roadmap
   https://www.youtube.com/results?search_query=lofi+beats


## 8. Running it for real

```bash
cd Voice_CLI_Virtual_Assistant
pip install -r requirements.txt

python main.py                       # interactive text session
python main.py --voice               # interactive voice session (needs a microphone)
python main.py --say "weather"       # one-shot command, handy in scripts
python main.py --list-commands       # print the whole command surface
python main.py --allow-power         # enable guarded shutdown/restart/logout
```

Run the test suite (43 tests, fully offline) with:

```bash
python -m unittest discover -s tests -v
```

A recorded terminal session covering every command — including live weather, live news and
a real Wikipedia lookup — is checked in at [`assets/demo_session.txt`](assets/demo_session.txt).